# Virtual screening with a 2D CNN on structure images

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yboulaamane/comp_chem_colab/blob/main/cnn_images_vs.ipynb)

The same task as the fingerprint notebook, separating MAO-B inhibitors from inactive
compounds in ChEMBL, but the model sees a picture of each structure, as a chemist would.
Each compound is drawn by RDKit as a 200 &times; 200 image and classified by a 2D
convolutional neural network. A random forest on fingerprints, on the same split, is the
baseline.

Runtime: GPU (Runtime &rarr; Change runtime type). On a CPU, training is slow.

## Setup

RDKit is not preinstalled on Colab; pandas, NumPy, scikit-learn, Matplotlib and TensorFlow are.

In [ ]:
%pip install -q rdkit

## Data: MAO-B inhibitors from ChEMBL

The compounds and their activities come from [ChEMBL](https://www.ebi.ac.uk/chembl/)
target [CHEMBL2039](https://www.ebi.ac.uk/chembl/explore/target/CHEMBL2039),
human monoamine oxidase B. The cell below downloads every activity record for
that target through the ChEMBL web services and turns them into one labelled row
per compound:

* Only IC50 and Ki values in nM that ChEMBL has not flagged as doubtful are used.
* Salts and counter-ions are stripped, so the different salt forms of one compound count once.
* A compound is **active (1)** if its median value is at most 316 nM (pIC50 &ge; 6.5) and
  **inactive (0)** if it is at least 10 &micro;M (pIC50 &le; 5). A compound reported only as
  "> 10 &micro;M" is also inactive. Compounds in between are left out.

The labelled table is saved to `maob_chembl_labelled.csv` and later runs read that file
instead of downloading again. Delete it to refresh the data. ChEMBL data is licensed under
[CC BY-SA 3.0](https://creativecommons.org/licenses/by-sa/3.0/).

In [ ]:
TARGET = "CHEMBL2039"   # monoamine oxidase B, Homo sapiens
ACTIVE_NM = 316         # median IC50 or Ki at or below this: active   (pIC50 >= 6.5)
INACTIVE_NM = 10_000    # median IC50 or Ki at or above this: inactive (pIC50 <= 5)
DATA_FILE = "maob_chembl_labelled.csv"

In [ ]:
import os
import time
from urllib.parse import urljoin

import numpy as np
import pandas as pd
import requests
from rdkit import Chem, RDLogger
from rdkit.Chem.MolStandardize import rdMolStandardize

RDLogger.DisableLog("rdApp.*")
CHEMBL_URL = "https://www.ebi.ac.uk"


def fetch_activities(target):
    """Every activity record ChEMBL holds for one target, 1000 per request."""
    url = f"{CHEMBL_URL}/chembl/api/data/activity.json"
    params = {"target_chembl_id": target, "limit": 1000}
    records = []
    while url:
        for attempt in range(5):
            try:
                response = requests.get(url, params=params, timeout=120)
                response.raise_for_status()
                break
            except requests.RequestException:
                if attempt == 4:
                    raise
                time.sleep(2 ** attempt)
        page = response.json()
        records += page["activities"]
        print(f"\rDownloaded {len(records)} of {page['page_meta']['total_count']} records", end="")
        next_page = page["page_meta"]["next"]
        url, params = (urljoin(CHEMBL_URL, next_page), None) if next_page else (None, None)
    print()
    if not records:
        raise ValueError(f"ChEMBL returned no activity records for {target}")
    return pd.DataFrame(records)


def parent_smiles(smiles):
    """Canonical SMILES of the largest fragment, neutralised, or None if RDKit cannot read it."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    try:
        return Chem.MolToSmiles(rdMolStandardize.ChargeParent(mol))
    except Exception:
        return None


def label_molecules(activities):
    """One row per parent structure: its first ChEMBL ID, SMILES and label (1 active, 0 inactive)."""
    a = activities.copy()
    a["relation"] = a["standard_relation"].fillna("").str.strip("'")
    a["value"] = pd.to_numeric(a["standard_value"], errors="coerce")
    usable = (a["standard_type"].isin(["IC50", "Ki"]) & (a["standard_units"] == "nM")
              & a["data_validity_comment"].isna() & a["canonical_smiles"].notna()
              & (a["value"] > 0))
    a = a[usable].copy()
    a["smiles"] = a["canonical_smiles"].map(parent_smiles)
    a = a[a["smiles"].notna()]

    median = a[a["relation"] == "="].groupby("smiles")["value"].median()
    labels = pd.Series(np.nan, index=median.index)
    labels[median <= ACTIVE_NM] = 1
    labels[median >= INACTIVE_NM] = 0

    # Compounds with no exact value, only "> x" with x of at least INACTIVE_NM
    above = a[a["relation"].isin([">", ">="]) & (a["value"] >= INACTIVE_NM)]["smiles"]
    only_above = pd.Index(above.unique()).difference(median.index)
    labels = pd.concat([labels, pd.Series(0.0, index=only_above)]).dropna().astype(int)

    first_id = a.sort_values("molecule_chembl_id").groupby("smiles")["molecule_chembl_id"].first()
    data = pd.DataFrame({"molecule_chembl_id": first_id[labels.index].to_numpy(),
                         "smiles": labels.index.to_numpy(),
                         "label": labels.to_numpy()})
    return data.sort_values("molecule_chembl_id", ignore_index=True)

In [ ]:
if os.path.exists(DATA_FILE):
    data = pd.read_csv(DATA_FILE)
else:
    data = label_molecules(fetch_activities(TARGET))
    data.to_csv(DATA_FILE, index=False)

n_active = int(data["label"].sum())
print(f"{len(data)} compounds: {n_active} active, {len(data) - n_active} inactive")
data.head()

## Structure images

Each image is built in memory in the same order as the table, so the images and labels
stay aligned. They are stored as 8-bit integers to save memory and scaled to 0&ndash;1 inside the model.

In [ ]:
import matplotlib.pyplot as plt
from rdkit.Chem import Draw

IMG_SIZE = 200


def structure_image(smiles):
    image = Draw.MolToImage(Chem.MolFromSmiles(smiles), size=(IMG_SIZE, IMG_SIZE))
    return np.asarray(image.convert("RGB"), dtype=np.uint8)


images = np.stack([structure_image(s) for s in data["smiles"]])
print(images.shape, f"{images.nbytes / 1e6:.0f} MB")

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for ax, i in zip(axes.flat, range(10)):
    ax.imshow(images[i])
    ax.set_title(f"{data['molecule_chembl_id'][i]}\n{'active' if data['label'][i] else 'inactive'}", fontsize=9)
    ax.axis("off")
plt.show()

## Train, validation and test sets

70 % of the compounds train the model, 10 % pick when to stop training and 20 % are kept
aside until the end. The split is stratified, so each set has the same share of actives.

A random split puts close analogues of training compounds in the test set, so the scores
below are optimistic for truly new chemistry. Splitting by scaffold gives a harder and more
realistic test.

In [ ]:
from sklearn.model_selection import train_test_split

y = data["label"].to_numpy()
idx_train, idx_test = train_test_split(np.arange(len(data)), test_size=0.2, stratify=y, random_state=42)
idx_train, idx_valid = train_test_split(idx_train, test_size=0.125, stratify=y[idx_train], random_state=42)

for name, idx in [("train", idx_train), ("validation", idx_valid), ("test", idx_test)]:
    print(f"{name:>10}: {len(idx):5d} compounds, {y[idx].mean():.0%} active")

In [ ]:
X_train, X_valid, X_test = images[idx_train], images[idx_valid], images[idx_test]
y_train, y_valid, y_test = y[idx_train], y[idx_valid], y[idx_test]

The classes are unbalanced. Instead of copying images, each class is weighted in the loss
in inverse proportion to its size in the training set.

In [ ]:
counts = np.bincount(y_train)
class_weight = {c: len(y_train) / (2 * n) for c, n in enumerate(counts)}
class_weight

## Model

Three convolution and pooling blocks, then a small dense layer and one sigmoid output:
the probability that the compound is active.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

keras.utils.set_random_seed(42)

model = keras.Sequential([
    keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    layers.Rescaling(1 / 255),
    layers.Conv2D(32, 3, activation="relu"),
    layers.MaxPooling2D(2),
    layers.Conv2D(64, 3, activation="relu"),
    layers.MaxPooling2D(2),
    layers.Conv2D(128, 3, activation="relu"),
    layers.MaxPooling2D(2),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])
model.compile(loss="binary_crossentropy", optimizer="adam",
              metrics=["accuracy", keras.metrics.AUC(name="auc")])
model.summary()

Training stops when the validation loss has not improved for 5 epochs and keeps the best
weights. The history is also written to `maob_cnn_images_training.csv`.

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, min_delta=1e-4, restore_best_weights=True),
    keras.callbacks.CSVLogger("maob_cnn_images_training.csv"),
]
history = model.fit(X_train, y_train, validation_data=(X_valid, y_valid), epochs=50,
                    batch_size=32, class_weight=class_weight, callbacks=callbacks)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, metric in zip(axes, ["loss", "auc"]):
    ax.plot(history.history[metric], label="train")
    ax.plot(history.history[f"val_{metric}"], label="validation")
    ax.set(xlabel="Epoch", ylabel=metric)
    ax.legend()
plt.show()

## Test set

The output is one probability per compound, so a compound is predicted active when it is
at least 0.5. ROC AUC does not depend on that threshold; MCC and the report below do.

In [ ]:
from sklearn.metrics import (ConfusionMatrixDisplay, RocCurveDisplay, classification_report,
                             matthews_corrcoef, roc_auc_score)

p_test = model.predict(X_test, verbose=0).ravel()
pred_test = (p_test >= 0.5).astype(int)

print(f"ROC AUC: {roc_auc_score(y_test, p_test):.3f}")
print(f"MCC:     {matthews_corrcoef(y_test, pred_test):.3f}")
print(classification_report(y_test, pred_test, target_names=["inactive", "active"]))
ConfusionMatrixDisplay.from_predictions(y_test, pred_test, display_labels=["inactive", "active"]);

## Baseline: random forest on the same split

A random forest on the same fingerprints needs no tuning and trains in seconds. If the
neural network does not beat it on the same test set, the extra complexity is not paying off.

In [ ]:
from rdkit.Chem import rdFingerprintGenerator
from sklearn.ensemble import RandomForestClassifier

morgan = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
fp = np.array([morgan.GetFingerprintAsNumPy(Chem.MolFromSmiles(s)) for s in data["smiles"]],
              dtype=np.uint8)

rf = RandomForestClassifier(n_estimators=500, class_weight="balanced", n_jobs=-1, random_state=42)
rf.fit(fp[idx_train], y_train)
p_rf = rf.predict_proba(fp[idx_test])[:, 1]
print(f"Random forest ROC AUC: {roc_auc_score(y_test, p_rf):.3f}")

ax = plt.gca()
RocCurveDisplay.from_predictions(y_test, p_test, name="2D CNN", ax=ax)
RocCurveDisplay.from_predictions(y_test, p_rf, name="Random forest", ax=ax)
ax.plot([0, 1], [0, 1], "k--", linewidth=1)
plt.show()

## Next steps

* Tune the architecture and training settings with [KerasTuner](https://keras.io/keras_tuner/),
  scoring each trial on the validation set only.
* Augment the training images (rotations, flips): a molecule's identity does not depend
  on how it is drawn.

## Save the model

Saved in the Keras format and, on Colab, downloaded to your computer. Load it again with
`keras.models.load_model("maob_cnn_images.keras")`.

In [ ]:
model.save("maob_cnn_images.keras")

try:
    from google.colab import files
    files.download("maob_cnn_images.keras")
except ImportError:  # not running on Colab
    pass